## 1. Import Required Libraries

Install dependencies and import core libraries for model training, data processing, and utilities.

In [ ]:
# Standard library imports
import json
import logging
import os
import sys
import random
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

# Scientific computing and ML libraries
import numpy as np
import torch
import yaml
from datasets import Dataset
from torch.utils.data import DataLoader

# Transformers library (Hugging Face)
from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    get_linear_schedule_with_warmup,
)

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

print("✓ Libraries imported successfully")

## 2. Define Configuration and Utility Functions

Load configuration from YAML and define helper functions for data loading.

In [ ]:
def load_config(config_path: str) -> dict:
    """Load configuration from YAML file."""
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    
    # Ensure numeric values are properly typed
    numeric_keys = [
        "max_seq_length",
        "batch_size",
        "eval_batch_size",
        "num_epochs",
        "learning_rate",
        "warmup_steps",
        "weight_decay",
        "gradient_accumulation_steps",
        "dataset_fraction",
        "max_span_length",
        "confidence_threshold",
    ]
    
    for key in numeric_keys:
        if key in config:
            if key in ["learning_rate", "weight_decay", "confidence_threshold"]:
                config[key] = float(config[key])
            else:
                # dataset_fraction should be treated as float not int
                if key == "dataset_fraction":
                    config[key] = float(config[key])
                else:
                    config[key] = int(config[key])
    
    return config


def load_jsonl(file_path: str) -> List[Dict]:
    """Load JSONL file."""
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

print("✓ Utility functions defined")

## 3. Data Conversion: JSONL → QA Format

Convert clickbait spoiler examples to question-answering format:
- Question = clickbait post text
- Context = article paragraphs
- Answer = spoiler span with character positions

In [ ]:
def create_qa_examples(data: List[Dict], dataset_type: str = "train") -> Tuple[List[Dict], List[Dict]]:
    """
    Convert clickbait spoiler dataset to QA format.
    
    Args:
        data: List of sample dictionaries from JSONL
        dataset_type: 'train' or 'validation'
    
    Returns:
        Tuple of (examples, features) where examples are QA-style and features track metadata
    """
    examples = []
    features = []
    
    for sample in data:
        # Extract necessary information
        post_text = " ".join(sample.get("postText", []))
        target_paragraphs = sample.get("targetParagraphs", [])
        spoiler_text = sample.get("spoiler", [])
        spoiler_positions = sample.get("spoilerPositions", [])
        tags = sample.get("tags", [])
        
        if not target_paragraphs or not spoiler_text:
            continue
        
        # Concatenate all target paragraphs to form the context
        context = " ".join(target_paragraphs)
        
        # Use post text as the question
        question = post_text
        
        # Process each spoiler position
        for spoiler_idx, spoiler_pos_list in enumerate(spoiler_positions):
            for para_idx, char_pos in enumerate(spoiler_pos_list):
                start_char, end_char = char_pos
                
                # Get the actual spoiler text at this position
                if para_idx < len(target_paragraphs):
                    paragraph = target_paragraphs[para_idx]
                    actual_spoiler = paragraph[start_char:end_char]
                    
                    # Calculate offset in concatenated context
                    context_offset = sum(
                        len(target_paragraphs[i]) + 1 for i in range(para_idx)  # +1 for space
                    )
                    
                    example = {
                        "question": question,
                        "context": context,
                        "answer_text": actual_spoiler,
                        "answer_start": context_offset + start_char,
                        "id": f"{sample.get('uuid', '')}__{spoiler_idx}_{para_idx}",
                    }
                    
                    examples.append(example)
                    features.append({
                        "id": sample.get("uuid", ""),
                        "tags": tags,
                        "platform": sample.get("postPlatform", ""),
                    })
    
    logger.info(f"Created {len(examples)} examples from {len(data)} samples")
    return examples, features

print("✓ QA conversion function defined")

## 4. Feature Preparation: Tokenization and Span Labeling

Tokenize text and map character-level answer positions to token-level start/end indices.

In [ ]:
def prepare_train_features(examples: Dict, tokenizer, config: dict) -> Dict:
    """Prepare features for training."""
    max_seq_length = config.get("max_seq_length", 512)
    doc_stride = 128
    
    # Tokenize contexts and questions
    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=max_seq_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    
    # Get offset mapping
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")
    
    # Initialize start and end labels
    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []
    
    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_mapping[i]
        answers = examples["answer_start"][sample_idx]
        answer_text = examples["answer_text"][sample_idx]
        
        # If no answer, set positions to (0, 0)
        if answers == -1:
            tokenized_examples["start_positions"].append(0)
            tokenized_examples["end_positions"].append(0)
            continue
        
        start_char = answers
        end_char = start_char + len(answer_text)
        
        # Find token positions
        token_start_index = 0
        token_end_index = len(offsets) - 1
        
        for j, (offset_start, offset_end) in enumerate(offsets):
            if offset_start <= start_char < offset_end:
                token_start_index = j
            if offset_start < end_char <= offset_end:
                token_end_index = j
                break
        
        tokenized_examples["start_positions"].append(token_start_index)
        tokenized_examples["end_positions"].append(token_end_index)
    
    return tokenized_examples


def prepare_validation_features(examples: Dict, tokenizer, config: dict) -> Dict:
    """Prepare features for validation."""
    max_seq_length = config.get("max_seq_length", 512)
    doc_stride = 128
    
    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=max_seq_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    
    # Store mapping info for post-processing
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    tokenized_examples["example_id"] = []
    
    for i in range(len(tokenized_examples["input_ids"])):
        sample_idx = sample_mapping[i]
        tokenized_examples["example_id"].append(examples["id"][sample_idx])
    
    return tokenized_examples

print("✓ Feature preparation functions defined")

## 5. Configuration Setup

Load training configuration from YAML file and check system capabilities (CUDA).

In [ ]:
# Set configuration path
CONFIG_PATH = "../configs/extractive_config.yaml"

# Load configuration
config = load_config(CONFIG_PATH)
print(f"\n📋 Configuration loaded from {CONFIG_PATH}")
print(f"Model: {config['model_name']}")
print(f"Batch size: {config['batch_size']}")
print(f"Learning rate: {config['learning_rate']}")
print(f"Num epochs: {config['num_epochs']}")
print(f"Dataset fraction: {config.get('dataset_fraction', 1.0)}")

## 6. Check System Resources

Verify CUDA availability and GPU properties.

In [ ]:
# Check CUDA availability and configuration
cuda_available = torch.cuda.is_available()
device_count = torch.cuda.device_count()

print("\n" + "="*60)
print("CUDA/Device Information")
print("="*60)
print(f"CUDA Available: {cuda_available}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"GPU Device Count: {device_count}")

if cuda_available:
    for i in range(device_count):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name}")
        print(f"  Memory: {props.total_memory / 1e9:.2f} GB")
else:
    print("\n⚠️ CUDA is not available. Training will use CPU.")
    print("To enable CUDA, reinstall PyTorch with CUDA support:")
    print("  pip install torch --index-url https://download.pytorch.org/whl/cu118")

print("="*60)

## 7. Load and Prepare Data

Load training and validation data from JSONL files and convert to QA format.

In [ ]:
# Create output directories
os.makedirs(config["output_dir"], exist_ok=True)
os.makedirs(config["checkpoint_dir"], exist_ok=True)
os.makedirs(config["log_dir"], exist_ok=True)

# Load data
data_dir = config["data_dir"]
train_file = os.path.join(data_dir, config["train_file"])
val_file = os.path.join(data_dir, config["validation_file"])

print(f"\n📂 Loading datasets...")
print(f"Training file: {train_file}")
print(f"Validation file: {val_file}")

train_data = load_jsonl(train_file)
train_examples, train_features = create_qa_examples(train_data, "train")

val_data = load_jsonl(val_file)
val_examples, val_features = create_qa_examples(val_data, "validation")

print(f"\n✓ Dataset loading complete")
print(f"  Train examples: {len(train_examples)}")
print(f"  Val examples: {len(val_examples)}")

## 8. Dataset Subsampling (Optional)

Subsample datasets if configured (e.g., 0.05 for 5% of data). Useful for quick testing.

In [ ]:
# Subsample datasets if configured (e.g., 0.05 for 5%)
dataset_fraction = config.get("dataset_fraction", 1.0)
if 0.0 < dataset_fraction < 1.0:
    seed = config.get("seed", 42)
    random.seed(seed)

    n_train = max(1, int(len(train_examples) * dataset_fraction))
    n_val = max(1, int(len(val_examples) * dataset_fraction))

    print(f"\n🔽 Subsampling datasets to {dataset_fraction*100:.2f}%")
    print(f"  Original train: {len(train_examples)} -> {n_train}")
    print(f"  Original val: {len(val_examples)} -> {n_val}")

    # If dataset is small enough that sample would be the same size, skip
    if n_train < len(train_examples):
        train_examples = random.sample(train_examples, n_train)
    if n_val < len(val_examples):
        val_examples = random.sample(val_examples, n_val)
else:
    print(f"\n✓ Using full dataset (fraction = {dataset_fraction})")

## 9. Load Model and Tokenizer

Load the pretrained DeBERTa-v3-base model and corresponding tokenizer.

In [ ]:
print(f"\n🔄 Loading model and tokenizer...")
print(f"Model: {config['model_name']}")

try:
    tokenizer = AutoTokenizer.from_pretrained(config["model_name"])
    model = AutoModelForQuestionAnswering.from_pretrained(config["model_name"])
    print("✓ Model loaded successfully")
except Exception as e:
    print(f"⚠️ Error loading model: {e}")
    print("Attempting with trust_remote_code=True...")
    tokenizer = AutoTokenizer.from_pretrained(config["model_name"], trust_remote_code=True)
    model = AutoModelForQuestionAnswering.from_pretrained(config["model_name"], trust_remote_code=True)
    print("✓ Model loaded with trust_remote_code=True")

## 10. Convert to HuggingFace Dataset Format

Convert QA examples to HuggingFace Dataset objects for efficient processing.

In [ ]:
print(f"\n📦 Creating HuggingFace Datasets...")

train_dataset = Dataset.from_dict({
    "question": [ex["question"] for ex in train_examples],
    "context": [ex["context"] for ex in train_examples],
    "answer_text": [ex["answer_text"] for ex in train_examples],
    "answer_start": [ex["answer_start"] for ex in train_examples],
    "id": [ex["id"] for ex in train_examples],
})

val_dataset = Dataset.from_dict({
    "question": [ex["question"] for ex in val_examples],
    "context": [ex["context"] for ex in val_examples],
    "answer_text": [ex["answer_text"] for ex in val_examples],
    "answer_start": [ex["answer_start"] for ex in val_examples],
    "id": [ex["id"] for ex in val_examples],
})

print(f"✓ Datasets created")
print(f"  Train: {len(train_dataset)} examples")
print(f"  Val: {len(val_dataset)} examples")

## 11. Tokenize and Prepare Features

Tokenize examples and map character positions to token positions for training.

In [ ]:
print(f"\n🔤 Tokenizing and preparing features...")
print(f"Max sequence length: {config['max_seq_length']}")

print("Preparing training features...")
train_dataset = train_dataset.map(
    lambda x: prepare_train_features(x, tokenizer, config),
    batched=True,
    remove_columns=train_dataset.column_names,
    batch_size=1000,
)

print("Preparing validation features...")
val_dataset = val_dataset.map(
    lambda x: prepare_validation_features(x, tokenizer, config),
    batched=True,
    remove_columns=val_dataset.column_names,
    batch_size=1000,
)

print(f"✓ Features prepared")
print(f"  Train: {len(train_dataset)} tokenized examples")
print(f"  Val: {len(val_dataset)} tokenized examples")

## 12. Configure Training Arguments

Set up training hyperparameters and device configuration.

In [ ]:
# Determine device and mixed precision settings
use_cuda = cuda_available and device_count > 0
use_fp16 = use_cuda  # Only use FP16 if CUDA is available

print(f"\n⚙️  Training Configuration")
print(f"Device: {'CUDA' if use_cuda else 'CPU'}")
print(f"Mixed Precision (FP16): {use_fp16}")
print(f"Batch size: {config['batch_size']}")
print(f"Learning rate: {config['learning_rate']}")
print(f"Epochs: {config['num_epochs']}")
print(f"Warmup steps: {config['warmup_steps']}")

# Set up training arguments
training_args = TrainingArguments(
    output_dir=config["output_dir"],
    eval_strategy="epoch",
    learning_rate=config["learning_rate"],
    per_device_train_batch_size=config["batch_size"],
    per_device_eval_batch_size=config["eval_batch_size"],
    num_train_epochs=config["num_epochs"],
    weight_decay=config["weight_decay"],
    warmup_steps=config["warmup_steps"],
    logging_dir=config["log_dir"],
    logging_steps=100,
    save_strategy="epoch",
    load_best_model_at_end=True,
    seed=42,
    fp16=use_fp16,
    no_cuda=not use_cuda,
)

print(f"\n✓ Training arguments configured")

## 13. Initialize Trainer

Create the Hugging Face Trainer object with model, datasets, and training arguments.

In [ ]:
print(f"\n🔧 Initializing Trainer...")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
)

print(f"✓ Trainer initialized and ready for training")

## 14. Train the Model

Start training! The model will be trained for the specified number of epochs with periodic validation.

In [ ]:
print(f"\n🚀 Starting training...")
print(f"Total steps: {len(train_dataset) // config['batch_size'] * config['num_epochs']}")

trainer.train()

print(f"\n✓ Training complete!")

## 15. Save the Trained Model

Save the final trained model and tokenizer to disk for future inference.

In [ ]:
print(f"\n💾 Saving model...")

model_save_path = os.path.join(config["output_dir"], "final_model")
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"✓ Model saved to: {model_save_path}")
print(f"\n📝 Model files:")
for file in os.listdir(model_save_path):
    print(f"  - {file}")

## 16. Summary and Next Steps

Training complete! Here's what to do next:

In [ ]:
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"\n✓ Model successfully trained and saved!")
print(f"\nModel location: {model_save_path}")
print(f"\nNext steps:")
print(f"  1. Use src/extractive_predictor.py to load and run inference")
print(f"  2. Evaluate model performance on test set")
print(f"  3. Fine-tune hyperparameters if needed")
print(f"  4. Proceed with abstractive refinement (FLAN-T5)")
print(f"\nTraining outputs:")
print(f"  - Model: {model_save_path}")
print(f"  - Checkpoints: {config['checkpoint_dir']}")
print(f"  - Logs: {config['log_dir']}")
print("="*60)